# Transformación de datos — demanda por tramo horario

Objetivo: construir, a partir de `informe_NOUP.csv`, un dataset limpio con el **target de
nº de citas por tramo horario y día** (mañana: antes de las 14:00, tarde: 14:00 en adelante),
sin desglosar por producto (se deja para una segunda fase).

Este notebook cubre la fase de **Transformación / Data Understanding**: deja el dataset
listo y ya dividido en train/test para que la fase de **EDA** (`eda.ipynb`) y la de
**Feature Engineering** (`feature_engineering.ipynb`) trabajen únicamente sobre el
histórico cerrado, sin tocar el conjunto de test.

Pasos:
1. Cargar el CSV crudo y descartar la fila de totales.
2. Parsear `Disponibilidad` en fecha de la cita + hora de inicio.
3. Quedarnos con reservas de servicio (excluir tarjetas de regalo / membresías) y deduplicar a nivel de reserva.
4. Quedarnos con las reservas confirmadas (no canceladas).
5. Aplicar un corte temporal fijo: solo citas ejecutadas hasta el 30/06/2026 (inclusive), para no meter en el modelo demanda todavía en curso o futura.
6. Asignar el tramo horario (mañana / tarde) según la hora de inicio.
7. Construir la rejilla completa fecha × tramo y agregar el nº de citas (target).
8. Añadir variables de calendario básicas (día de la semana, mes, fin de semana...).
9. Comprobaciones de calidad.
10. Guardado del dataset completo.
11. División train / test **cronológica** (antes de la EDA, para evitar data leakage).

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)

RAW_PATH = Path('informe_NOUP.csv')
OUTPUT_PATH = Path('data/processed/ocupacion_tramos.csv')
TRAIN_PATH = Path('data/processed/train.csv')
TEST_PATH = Path('data/processed/test.csv')

# Productos que no representan una cita con horario (bonos / financieros)
NON_SERVICE_PRODUCTS = ['¡Tarjeta de regalo!', 'Membresías']

# Hora de corte entre el tramo "mañana" y el tramo "tarde"
CORTE_TARDE_MIN = 14 * 60  # 14:00 en minutos desde medianoche

# Solo se usan citas ejecutadas hasta esta fecha (inclusive): evita meter en el
# modelo demanda todavía en curso (reservas futuras o muy recientes al corte de datos)
FECHA_CORTE = pd.Timestamp('2026-06-30')

## 1. Carga del CSV crudo

In [2]:
df_raw = pd.read_csv(RAW_PATH, skiprows=1, encoding='utf-8-sig')

# La última fila es un resumen de totales del propio export
# ("6487 disponibilidades, 22 productos, 13 tipos de pago...") y no es una reserva real
df_raw = df_raw[df_raw['ID de reserva'].notna()].copy()

print(f"Filas: {len(df_raw)} | Reservas únicas: {df_raw['ID de reserva'].nunique()}")
df_raw.head(3)

Filas: 9081 | Reservas únicas: 8106


,Disponibilidad,Producto,Tipo de pago,Creado el,Creado en hora,Creado en fecha,Pago o reembolso,ID de pago o reembolso,Creado por,Tipo de pago.1,Últim. 4 dígitos tarj. créd.,Código postal de tarjeta de crédito,Tipo de tarjeta regalo,Bruto,Cargo de gestión,Neto,Pago bruto,Pago neto,Impuesto pagado,ID de abono,Fecha de abono,ID de reserva,ID de pedido,¿Cancelado?,Reprogramación,Creado el.1,Creado por.1,Cancelado el,ID de producto,Producto.1,Disponibilidad.1,Contacto,Teléfono,Idioma de contacto,País del teléfono,Email,Referencia de reserva online,Total
0,3/5/24,¡Tarjeta de regalo!,Tpv,2024-05-16 a las 14:03,14:03,2024-05-16,Payment,#165831557,Recepcion Oasis,TPV,NaN,NaN,NaN,"45,00 €","0,00 €","45,00 €","45,00 €","45,00 €","0,00 €",NaN,NaN,#223688445,NaN,No,No,2024-05-16 a las 14:03,Recepcion Oasis,NaN,#545896,¡Tarjeta de regalo!,2024-05-03 a las 00:00,Mar Rodriguez,687 37 60 68,Español,ES,NaN,NaN,"45,00 €"
1,4/5/24,¡Tarjeta de regalo!,Cash,2024-05-16 a las 10:01,10:01,2024-05-16,Payment,#165820811,Recepcion Oasis,Cash,NaN,NaN,NaN,"45,00 €","0,00 €","45,00 €","45,00 €","45,00 €","0,00 €",NaN,NaN,#223670538,NaN,No,No,2024-05-16 a las 10:01,Recepcion Oasis,NaN,#545896,¡Tarjeta de regalo!,2024-05-04 a las 00:00,Josue Torres Santiago,662 22 05 24,Español,ES,NaN,NaN,"45,00 €"
2,4/5/24,¡Tarjeta de regalo!,Cash,2024-05-16 a las 10:10,10:10,2024-05-16,Payment,#165821212,Recepcion Oasis,Cash,NaN,NaN,NaN,"45,00 €","0,00 €","45,00 €","45,00 €","45,00 €","0,00 €",NaN,NaN,#223671146,NaN,No,No,2024-05-16 a las 10:10,Recepcion Oasis,NaN,#545896,¡Tarjeta de regalo!,2024-05-04 a las 00:00,Pedro Romero,671 38 28 19,Español,ES,oasissevilla@hotmail.com,NaN,"45,00 €"


## 2. Parseo de `Disponibilidad`: fecha y hora de la cita

`Disponibilidad` mezcla varios formatos: `"9/5/24 a las 12:00 – 13:40"` (rango),
`"9/5/24 a las 20:00"` (solo inicio) y `"3/5/24"` (sin hora — bonos, se descartan más adelante).

In [3]:
DISP_RE = re.compile(r'^(\d{1,2}/\d{1,2}/\d{2,4})(?:\s+a las\s+(\d{1,2}:\d{2})(?:\s*[-–—]\s*(\d{1,2}:\d{2}))?)?')


def parse_disponibilidad(valor):
    if pd.isna(valor):
        return pd.NaT, None, None
    m = DISP_RE.match(str(valor))
    if not m:
        return pd.NaT, None, None
    fecha_str, hora_inicio, hora_fin = m.groups()
    try:
        fecha = pd.to_datetime(fecha_str, format='%d/%m/%y')
    except ValueError:
        fecha = pd.NaT
    return fecha, hora_inicio, hora_fin


parsed = df_raw['Disponibilidad'].apply(parse_disponibilidad)
df_raw['fecha_cita'] = parsed.apply(lambda x: x[0])
df_raw['hora_inicio'] = parsed.apply(lambda x: x[1])
df_raw['hora_fin'] = parsed.apply(lambda x: x[2])

print(f"Filas sin fecha_cita parseable: {df_raw['fecha_cita'].isna().sum()}")

Filas sin fecha_cita parseable: 0


## 3. Reservas de servicio, deduplicadas

Cada reserva puede tener varias filas (pago + reembolso, pago dividido...). Se excluyen
tarjetas de regalo / membresías (no son citas con horario) y se deduplica a nivel de
`ID de reserva`, quedándonos con la transacción más antigua de cada una.

In [4]:
servicios = df_raw[~df_raw['Producto'].isin(NON_SERVICE_PRODUCTS)].copy()

reservas = (
    servicios.sort_values('Creado el')
             .drop_duplicates(subset='ID de reserva', keep='first')
             .copy()
)

print(f"Reservas de servicio (dedup): {len(reservas)}")
reservas['¿Cancelado?'].value_counts(dropna=False)

Reservas de servicio (dedup): 6190


¿Cancelado?
No           6101
Cancelled      89
Name: count, dtype: int64

## 4. Solo reservas confirmadas (no canceladas)

El target debe reflejar demanda real atendida, así que se excluyen las reservas `Cancelled`
(~1,4% del total) y cualquier fila sin fecha/hora de cita parseada.

In [5]:
confirmadas = reservas[reservas['¿Cancelado?'] == 'No'].copy()
antes = len(confirmadas)
confirmadas = confirmadas.dropna(subset=['fecha_cita', 'hora_inicio'])

print(f"Reservas confirmadas: {antes} | con cita válida (fecha + hora): {len(confirmadas)}")

Reservas confirmadas: 6101 | con cita válida (fecha + hora): 6101


## 5. Corte temporal: solo citas ejecutadas hasta el 30/06/2026 (inclusive)

Para no meter en el modelo demanda todavía en curso (reservas de fechas futuras, o de
fechas muy recientes cuyo recuento aún puede crecer porque siguen llegando reservas), se
descarta cualquier cita posterior a `FECHA_CORTE`. Todo lo que viene después (rejilla,
target, features) se construye solo con este histórico ya cerrado.

In [6]:
antes = len(confirmadas)
confirmadas = confirmadas[confirmadas['fecha_cita'] <= FECHA_CORTE].copy()

print(f"Reservas confirmadas antes del corte temporal: {antes}")
print(f"Reservas confirmadas hasta {FECHA_CORTE.date()} (inclusive): {len(confirmadas)}")
print(f"Descartadas por ser posteriores al corte: {antes - len(confirmadas)}")

Reservas confirmadas antes del corte temporal: 6101
Reservas confirmadas hasta 2026-06-30 (inclusive): 6040
Descartadas por ser posteriores al corte: 61


## 6. Tramo horario (mañana / tarde, corte a las 14:00)

In [7]:
def minutos_desde_medianoche(hora_str):
    h, m = hora_str.split(':')
    return int(h) * 60 + int(m)


confirmadas['hora_inicio_min'] = confirmadas['hora_inicio'].apply(minutos_desde_medianoche)
confirmadas['tramo'] = np.where(confirmadas['hora_inicio_min'] < CORTE_TARDE_MIN, 'mañana', 'tarde')

confirmadas.groupby('tramo').size()

tramo
mañana    2197
tarde     3843
dtype: int64

## 7. Rejilla fecha × tramo y target (nº de citas)

Se construye el calendario completo desde la primera cita hasta `FECHA_CORTE`, cruzado con
los dos tramos, para que los días/tramos sin ninguna reserva queden explícitos como `0` en
vez de faltar en el dataset.

In [8]:
conteo = confirmadas.groupby(['fecha_cita', 'tramo']).size().rename('n_citas').reset_index()

fechas = pd.date_range(confirmadas['fecha_cita'].min(), FECHA_CORTE, freq='D')
tramos = ['mañana', 'tarde']
rejilla = pd.MultiIndex.from_product([fechas, tramos], names=['fecha_cita', 'tramo']).to_frame(index=False)

dataset = rejilla.merge(conteo, on=['fecha_cita', 'tramo'], how='left')
dataset['n_citas'] = dataset['n_citas'].fillna(0).astype(int)

assert dataset['n_citas'].sum() == len(confirmadas)
dataset.head()

,fecha_cita,tramo,n_citas
0,2024-05-09,mañana,0
1,2024-05-09,tarde,3
2,2024-05-10,mañana,1
3,2024-05-10,tarde,2
4,2024-05-11,mañana,2


## 8. Variables de calendario

Estas son la **descomposición directa** de `fecha_cita` — no vienen de ningún análisis,
solo de partir la fecha en sus componentes. Se calculan aquí, antes del split train/test,
porque la EDA (`eda.ipynb`) las necesita para poder agrupar y graficar desde el primer
momento, sin tener que re-derivarlas. Las variables *derivadas con criterio* (a partir de
lo que muestre la EDA) van en `feature_engineering.ipynb`, no aquí — ver la nota al
principio de ese notebook para la distinción completa.

*(Nota de redundancia: `dia_semana` (0-6) y `nombre_dia` (texto) son la misma información
en dos formatos — se mantienen las dos porque `nombre_dia` hace mucho más legibles los
gráficos y tablas de la EDA, pero para el modelo basta con `dia_semana`; `nombre_dia` es
candidata a eliminarse en Preprocesado, igual que pasará con `tramo` frente a
`tramo_tarde` en Feature Engineering.)*

In [9]:
dataset['dia_semana'] = dataset['fecha_cita'].dt.dayofweek  # 0 = lunes
dataset['nombre_dia'] = dataset['fecha_cita'].dt.day_name()
dataset['es_finde'] = dataset['dia_semana'].isin([5, 6])
dataset['mes'] = dataset['fecha_cita'].dt.month
dataset['anio'] = dataset['fecha_cita'].dt.year
dataset['semana_iso'] = dataset['fecha_cita'].dt.isocalendar().week.astype(int)

dataset = dataset.sort_values(['fecha_cita', 'tramo']).reset_index(drop=True)
dataset.head()

,fecha_cita,tramo,n_citas,dia_semana,nombre_dia,es_finde,mes,anio,semana_iso
0,2024-05-09,mañana,0,3,Thursday,False,5,2024,19
1,2024-05-09,tarde,3,3,Thursday,False,5,2024,19
2,2024-05-10,mañana,1,4,Friday,False,5,2024,19
3,2024-05-10,tarde,2,4,Friday,False,5,2024,19
4,2024-05-11,mañana,2,5,Saturday,True,5,2024,19


## 9. Comprobaciones de calidad

In [10]:
print("Rango de fechas:", dataset['fecha_cita'].min().date(), "->", dataset['fecha_cita'].max().date())
print("Filas totales (fecha x tramo):", len(dataset))
print("\nNulos por columna:")
print(dataset.isna().sum())

print("\nTotal de citas en el dataset transformado:", dataset['n_citas'].sum())
print("Total de reservas confirmadas de origen (hasta el corte):", len(confirmadas))

print("\nMedia de n_citas por tramo:")
print(dataset.groupby('tramo')['n_citas'].mean())

Rango de fechas: 2024-05-09 -> 2026-06-30
Filas totales (fecha x tramo): 1566

Nulos por columna:
fecha_cita    0
tramo         0
n_citas       0
dia_semana    0
nombre_dia    0
es_finde      0
mes           0
anio          0
semana_iso    0
dtype: int64

Total de citas en el dataset transformado: 6040
Total de reservas confirmadas de origen (hasta el corte): 6040

Media de n_citas por tramo:
tramo
mañana    2.805875
tarde     4.908046
Name: n_citas, dtype: float64


## 10. Guardado del dataset transformado

In [11]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
dataset.to_csv(OUTPUT_PATH, index=False)
print(f"Guardado en: {OUTPUT_PATH.resolve()}")
dataset.tail()

Guardado en: D:\Usuarios\Emilio\Documentos\GitHub\ML_Spa\ML_Spa_M003\data\processed\ocupacion_tramos.csv


,fecha_cita,tramo,n_citas,dia_semana,nombre_dia,es_finde,mes,anio,semana_iso
1561,2026-06-28,tarde,7,6,Sunday,True,6,2026,26
1562,2026-06-29,mañana,3,0,Monday,False,6,2026,27
1563,2026-06-29,tarde,6,0,Monday,False,6,2026,27
1564,2026-06-30,mañana,2,1,Tuesday,False,6,2026,27
1565,2026-06-30,tarde,4,1,Tuesday,False,6,2026,27


## 11. División train / test (split cronológico)

La guía del proyecto pide hacer el split train/test **antes** de la EDA, para no explorar
ni tomar decisiones contaminadas por el conjunto de test. Al ser una serie temporal, el
split **no puede ser aleatorio** (mezclar fechas al azar filtraría información del futuro
al pasado, algo que en producción nunca tendríamos) — se reserva como test el tramo
cronológico más reciente, respetando el orden temporal. Se mantiene el mismo `test_size`
de referencia que usa la guía (20%), aplicado aquí sobre el eje temporal.

A partir de aquí, **`eda.ipynb` y `feature_engineering.ipynb` solo deben leer `train.csv`**;
`test.csv` se reserva para la evaluación final del modelo.

In [12]:
TEST_SIZE = 0.2  # mismo ratio que usa la guía para el split genérico, aplicado aquí de forma cronológica

fechas_unicas = dataset['fecha_cita'].drop_duplicates().sort_values().reset_index(drop=True)
n_test_dias = int(np.ceil(len(fechas_unicas) * TEST_SIZE))
fecha_split = fechas_unicas.iloc[-n_test_dias]

train = dataset[dataset['fecha_cita'] < fecha_split].copy()
test = dataset[dataset['fecha_cita'] >= fecha_split].copy()

print(f"Fecha de corte train/test: {fecha_split.date()}")
print(f"Train: {train['fecha_cita'].min().date()} -> {train['fecha_cita'].max().date()} "
      f"| {len(train)} filas ({train['fecha_cita'].nunique()} días)")
print(f"Test:  {test['fecha_cita'].min().date()} -> {test['fecha_cita'].max().date()} "
      f"| {len(test)} filas ({test['fecha_cita'].nunique()} días)")

train.to_csv(TRAIN_PATH, index=False)
test.to_csv(TEST_PATH, index=False)
print(f"\nGuardado train en: {TRAIN_PATH.resolve()}")
print(f"Guardado test en:  {TEST_PATH.resolve()}")

Fecha de corte train/test: 2026-01-25
Train: 2024-05-09 -> 2026-01-24 | 1252 filas (626 días)
Test:  2026-01-25 -> 2026-06-30 | 314 filas (157 días)

Guardado train en: D:\Usuarios\Emilio\Documentos\GitHub\ML_Spa\ML_Spa_M003\data\processed\train.csv
Guardado test en:  D:\Usuarios\Emilio\Documentos\GitHub\ML_Spa\ML_Spa_M003\data\processed\test.csv
